In [1]:
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

def generate_spectrogram(audio_path, output_path):
    y, sr = librosa.load(audio_path, sr=None)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    plt.figure(figsize=(2.24, 2.24))  # Size for 224x224 pixels
    librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel')
    plt.axis('off')
    plt.tight_layout()

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close()

def process_dataset(dataset_dir, output_dir):
    for label in ['normal', 'abnormal']:
        folder = os.path.join(dataset_dir, label)
        output_label_folder = os.path.join(output_dir, os.path.basename(dataset_dir), label)
        for filename in os.listdir(folder):
            if filename.endswith('.wav'):
                input_path = os.path.join(folder, filename)
                output_path = os.path.join(output_label_folder, filename.replace('.wav', '.png'))
                generate_spectrogram(input_path, output_path)

process_dataset('/content/dataset/id_02', '/content/spectrograms')
process_dataset('/content/dataset/id_04', '/content/spectrograms')


In [27]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define paths and parameters
dataset_dir = '/content/spectrograms/id_02'  # Use id_04 for the second dataset
img_height, img_width = 128, 128
batch_size = 32

# Create data generators with 80/20 split
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2  # 80% training, 20% validation
)

# Training set
train_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    shuffle=True
)

# Validation set
val_generator = datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)


Found 16 images belonging to 2 classes.
Found 4 images belonging to 2 classes.


In [32]:
# Paths
pretrain_path = '/content/spectrograms/id_02'
finetune_path = '/content/spectrograms/id_04'

# Parameters
img_size = (224, 224)
batch_size = 32
epochs_pretrain = 5
epochs_finetune = 5

# Data Generators
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

# Pre-training generators (id_02)
train_gen_02 = train_datagen.flow_from_directory(
    pretrain_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_gen_02 = train_datagen.flow_from_directory(
    pretrain_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

# Fine-tuning/Test generator (id_04)
test_gen_04 = test_datagen.flow_from_directory(
    finetune_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)


Found 16 images belonging to 2 classes.
Found 4 images belonging to 2 classes.
Found 20 images belonging to 2 classes.


In [25]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │     3,686,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,705,921 (14.14 MB)

 Trainable params: 3,705,921 (14.14 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.8125 - loss: 0.6611 - val_accuracy: 0.5000 - val_loss: 0.6834
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step - accuracy: 0.4375 - loss: 1.1948 - val_accuracy: 0.5000 - val_loss: 0.8820
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 606ms/step - accuracy: 0.5000 - loss: 1.0813 - val_accuracy: 0.5000 - val_loss: 0.7185
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 674ms/step - accuracy: 0.5625 - loss: 0.7137 - val_accuracy: 0.5000 - val_loss: 0.7794
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6875 - loss: 0.6100 - val_accuracy: 0.5000 - val_loss: 0.6747
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7500 - loss: 0.5287 - val_accuracy: 0.5000 - val_loss: 0.6928
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step - accuracy: 0.9375 - loss: 0.4078 - val_accuracy: 0.5000 - val_loss: 0.7107
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 0.8750 - loss: 0.5060 - val_accuracy: 0.5000 - val_loss: 0.7575
Epoch 9/10
1

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (224, 224)
batch_size = 16

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    '/content/spectrograms/id_02',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    '/content/spectrograms/id_02',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)


Found 16 images belonging to 2 classes.
Found 4 images belonging to 2 classes.


In [3]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, f1_score
import numpy as np

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base

model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_generator, validation_data=val_generator, epochs=5)

# Evaluate F1 score
y_true = val_generator.classes
y_pred = (model.predict(val_generator) > 0.5).astype('int32')
print("F1 Score:", f1_score(y_true, y_pred))


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 ━━━━━━━━━━━━━━━━━━━━ 16s 16s/step - accuracy: 0.3750 - loss: 0.9486 - val_accuracy: 0.5000 - val_loss: 0.7062
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6875 - loss: 0.6789 - val_accuracy: 0.5000 - val_loss: 0.7583
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.5625 - loss: 0.7111 - val_accuracy: 0.2500 - val_loss: 0.7680
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6875 - loss: 0.5713 - val_accuracy: 0.5000 - val_loss: 0.7679
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 21s 21s/step - accuracy: 0.6875 - loss: 0.6879 - val_accuracy: 0.2500 - val_loss: 0.7898
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
F1 Score: 0.4


In [34]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),  # Make sure this matches your actual image size
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')  # or softmax for multiclass
])


In [35]:
train_gen_02 = tf.keras.utils.image_dataset_from_directory(
    '/content/spectrograms/id_02',
    image_size=(128, 128),    # Must match model input
    batch_size=32,
    color_mode='rgb'          # or 'grayscale' if 1-channel
)


Found 20 files belonging to 2 classes.


In [36]:
for images, labels in train_gen_02.take(1):
    print(images.shape)  # e.g., (32, 128, 128, 3)


(20, 128, 128, 3)


In [40]:
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# Example of resizing an image to the required shape (224, 224)
def preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))  # Resize to (224, 224)
    img = img_to_array(img)  # Convert to array
    img = img / 255.0  # Normalize the image
    return img

In [41]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),  # Adjust input shape here
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')  # Assuming binary classification
])


In [10]:

train_accuracy = history.history['accuracy'][-1]
val_accuracy = history.history['val_accuracy'][-1]
print(f"Transfer Learning - Training Accuracy: {train_accuracy:.4f}")
print(f"Transfer Learning - Validation Accuracy: {val_accuracy:.4f}")


Transfer Learning - Training Accuracy: 0.6875
Transfer Learning - Validation Accuracy: 0.2500


In [4]:
model.save('/content/vgg_model_id02.h5')


In [5]:
from tensorflow.keras.models import load_model

model2 = load_model('/content/vgg_model_id02.h5')  # Load saved model
model2.trainable = True  # Fine-tune entire model

train2 = train_datagen.flow_from_directory(
    '/content/spectrograms/id_04',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val2 = train_datagen.flow_from_directory(
    '/content/spectrograms/id_04',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

model2.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

history2 = model2.fit(train2, validation_data=val2, epochs=5)

# F1 Score after transfer
y_true2 = val2.classes
y_pred2 = (model2.predict(val2) > 0.5).astype('int32')
print("F1 Score after transfer:", f1_score(y_true2, y_pred2))


Found 16 images belonging to 2 classes.
Found 4 images belonging to 2 classes.
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step - accuracy: 0.3750 - loss: 0.9099 - val_accuracy: 0.7500 - val_loss: 0.5890
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - accuracy: 0.6875 - loss: 0.5848 - val_accuracy: 1.0000 - val_loss: 0.5761
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 16s 16s/step - accuracy: 0.6875 - loss: 0.6292 - val_accuracy: 1.0000 - val_loss: 0.5638
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.5000 - loss: 0.7278 - val_accuracy: 1.0000 - val_loss: 0.5521
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 20s 20s/step - accuracy: 0.5000 - loss: 0.7439 - val_accuracy: 1.0000 - val_loss: 0.5409
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
F1 Score after transfer: 1.0


In [11]:
train_accuracy2 = history2.history['accuracy'][-1]
val_accuracy2 = history2.history['val_accuracy'][-1]
print(f"Fine-Tuned Transfer Learning - Training Accuracy: {train_accuracy2:.4f}")
print(f"Fine-Tuned Transfer Learning - Validation Accuracy: {val_accuracy2:.4f}")

Fine-Tuned Transfer Learning - Training Accuracy: 0.5000
Fine-Tuned Transfer Learning - Validation Accuracy: 1.0000


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.fit(train2, validation_data=val2, epochs=5)

y_pred_cnn = (cnn_model.predict(val2) > 0.5).astype('int32')
print("F1 Score without transfer:", f1_score(y_true2, y_pred_cnn))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.5000 - loss: 0.6979 - val_accuracy: 0.5000 - val_loss: 0.9969
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6250 - loss: 0.9673 - val_accuracy: 0.5000 - val_loss: 3.9101
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5625 - loss: 2.2864 - val_accuracy: 0.5000 - val_loss: 0.5987
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6250 - loss: 1.2315 - val_accuracy: 0.5000 - val_loss: 1.1669
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6875 - loss: 1.0842 - val_accuracy: 0.7500 - val_loss: 0.5805
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
F1 Score without transfer: 0.6666666666666666


In [7]:
import shutil
shutil.make_archive('/content/spectrograms', 'zip', '/content/spectrograms')


'/content/spectrograms.zip'

In [45]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Paths to the datasets
train_dir_02 = '/content/spectrograms/id_02'
val_dir_02 = '/content/spectrograms/id_02'

train_dir_04 = '/content/spectrograms/id_04'
val_dir_04 = '/content/spectrograms/id_04'

# Parameters
epochs_pretrain = 5
epochs_finetune = 5
batch_size = 32

# 1. Data Generators for both datasets
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen_02 = train_datagen.flow_from_directory(train_dir_02, target_size=(224, 224), batch_size=batch_size, class_mode='binary')
val_gen_02 = val_datagen.flow_from_directory(val_dir_02, target_size=(224, 224), batch_size=batch_size, class_mode='binary')

train_gen_04 = train_datagen.flow_from_directory(train_dir_04, target_size=(224, 224), batch_size=batch_size, class_mode='binary')
val_gen_04 = val_datagen.flow_from_directory(val_dir_04, target_size=(224, 224), batch_size=batch_size, class_mode='binary')

# 2. Define a simple CNN model
def create_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 3. Pre-train on id_02
print("==== Pre-training on id_02 ====")
model = create_model()
model.fit(train_gen_02, validation_data=val_gen_02, epochs=epochs_pretrain)

# 4. Fine-tune on id_04
print("==== Fine-tuning on id_04 ====")
# Unfreeze some layers (fine-tuning)
for layer in model.layers[:10]:  # Freeze first few layers (adjust as needed)
    layer.trainable = False

model.fit(train_gen_04, validation_data=val_gen_04, epochs=epochs_finetune)

# 5. Save the model after fine-tuning
model.save('finetuned_model.h5')


Found 20 images belonging to 2 classes.
Found 20 images belonging to 2 classes.
Found 20 images belonging to 2 classes.
Found 20 images belonging to 2 classes.
==== Pre-training on id_02 ====


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5500 - loss: 0.6881 - val_accuracy: 0.5000 - val_loss: 21.5837
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 21.5837 - val_accuracy: 0.5000 - val_loss: 2.2271
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 2.2271 - val_accuracy: 0.5000 - val_loss: 1.5793
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 1.5793 - val_accuracy: 0.5000 - val_loss: 1.9645
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 1.9645 - val_accuracy: 0.5000 - val_loss: 1.5535
==== Fine-tuning on id_04 ====
Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 2.3279 - val_accuracy: 0.5000 - val_loss: 1.0672
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 1.0672 - val_accuracy: 0.5000 - val_loss: 1.1813
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 1.1813 - val_accuracy: 0.6500 - val_loss: 0.6175
E

In [8]:
from google.colab import files
files.download('/content/spectrograms.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# Accuracy and F1 Score of Fine-Tuned Transfer Learning Model
fine_tuned_train_accuracy = history2.history['accuracy'][-1]
fine_tuned_val_accuracy = history2.history['val_accuracy'][-1]
fine_tuned_f1_score = f1_score(y_true2, y_pred2)

# Accuracy and F1 Score of CNN Model (Trained from Scratch)
cnn_train_accuracy = history.history['accuracy'][-1]
cnn_val_accuracy = history.history['val_accuracy'][-1]
cnn_f1_score = f1_score(y_true2, y_pred_cnn)

print("Fine-Tuned Transfer Learning - Validation Accuracy:", fine_tuned_val_accuracy)
print("CNN Without Transfer Learning - Validation Accuracy:", cnn_val_accuracy)

# Check for negative transfer
if fine_tuned_val_accuracy < cnn_val_accuracy:
    print("Negative Transfer Detected!")
else:
    print("No Negative Transfer Detected!")

print("\nFine-Tuned Transfer Learning - F1 Score:", fine_tuned_f1_score)
print("CNN Without Transfer Learning - F1 Score:", cnn_f1_score)

# Compare F1 Scores
if fine_tuned_f1_score < cnn_f1_score:
    print("Negative Transfer Detected Based on F1 Score!")
else:
    print("No Negative Transfer Detected Based on F1 Score!")


Fine-Tuned Transfer Learning - Validation Accuracy: 1.0
CNN Without Transfer Learning - Validation Accuracy: 0.25
No Negative Transfer Detected!

Fine-Tuned Transfer Learning - F1 Score: 1.0
CNN Without Transfer Learning - F1 Score: 0.6666666666666666
No Negative Transfer Detected Based on F1 Score!
